# structure of agent reply


In [193]:
from pydantic import BaseModel
from typing import List

class Task(BaseModel):
    id: int
    name: str
    description: str
    agent_type: str    
    url: str 

In [185]:
class TaskPlan(BaseModel):
    user_requirement: str
    tasks: List[Task]

## API keys

In [40]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [26]:
from google import genai

gemini_client = genai.Client(
    api_key=GOOGLE_API_KEY
)

print("Gemini client created")

Gemini client created


In [27]:
user_requirement = """
Find me the best laptop under ₹80,000
for programming, gaming and college use.
"""

In [17]:
orchestrator_prompt = f"""
You are the Orchestrator of a shopping research system.

Your job is to understand the user's requirement
and break it into research tasks.

You are NOT doing the research.

For each task determine:

- task id
- task name
- what the task should accomplish
- which type of agent should perform it

Available agent types:

browser
llm
logic

Use browser when information must be collected from websites.
Use llm for reasoning, summarization or analysis.
Use logic for deterministic calculations.

User requirement:

{user_requirement}
"""

## Orchestrator generator

In [30]:
response = gemini_client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=orchestrator_prompt,
    config={
        "response_mime_type": "application/json",
        "response_schema": TaskPlan.model_json_schema(),
    },
)

In [31]:
print(response.text)

{
  "user_requirement": "Find me the best laptop under ₹80,000 for programming, gaming and college use.",
  "tasks": [
    {
      "id": 1,
      "name": "Market Search",
      "description": "Search e-commerce websites and tech blogs to gather a list of laptops under ₹80,000 suitable for programming, gaming, and college use.",
      "agent_type": "browser"
    },
    {
      "id": 2,
      "name": "Feature Analysis",
      "description": "Analyze the specifications of the shortlisted laptops such as CPU, GPU, RAM, battery life, and build quality to determine their suitability for programming, gaming, and portability.",
      "agent_type": "llm"
    },
    {
      "id": 3,
      "name": "Score Calculation",
      "description": "Calculate a weighted score for each laptop based on performance, price, battery life, and portability to rank them objectively.",
      "agent_type": "logic"
    },
    {
      "id": 4,
      "name": "Final Recommendation",
      "description": "Summarize the f

## convert to python objs, not json
## also maintains the TaskPlan Schema


In [75]:
task_plan = TaskPlan.model_validate_json(response.text)

In [76]:
print(task_plan)

user_requirement='Find me the best laptop under ₹80,000 for programming, gaming and college use.' tasks=[Task(id=1, name='Market Search', description='Search e-commerce websites and tech blogs to gather a list of laptops under ₹80,000 suitable for programming, gaming, and college use.', agent_type='browser'), Task(id=2, name='Feature Analysis', description='Analyze the specifications of the shortlisted laptops such as CPU, GPU, RAM, battery life, and build quality to determine their suitability for programming, gaming, and portability.', agent_type='llm'), Task(id=3, name='Score Calculation', description='Calculate a weighted score for each laptop based on performance, price, battery life, and portability to rank them objectively.', agent_type='logic'), Task(id=4, name='Final Recommendation', description='Summarize the findings and generate a final recommendation report detailing the best laptop choices under ₹80,000.', agent_type='llm')]


In [37]:
import pprint
# type(task_plan)
pprint.pprint(task_plan)

TaskPlan(user_requirement='Find me the best laptop under ₹80,000 for programming, gaming and college use.', tasks=[Task(id=1, name='Market Search', description='Search e-commerce websites and tech blogs to gather a list of laptops under ₹80,000 suitable for programming, gaming, and college use.', agent_type='browser'), Task(id=2, name='Feature Analysis', description='Analyze the specifications of the shortlisted laptops such as CPU, GPU, RAM, battery life, and build quality to determine their suitability for programming, gaming, and portability.', agent_type='llm'), Task(id=3, name='Score Calculation', description='Calculate a weighted score for each laptop based on performance, price, battery life, and portability to rank them objectively.', agent_type='logic'), Task(id=4, name='Final Recommendation', description='Summarize the findings and generate a final recommendation report detailing the best laptop choices under ₹80,000.', agent_type='llm')])


In [38]:
for task in task_plan.tasks:
    print(
        task.id,
        "|",
        task.name,
        "|",
        task.agent_type
    )

1 | Market Search | browser
2 | Feature Analysis | llm
3 | Score Calculation | logic
4 | Final Recommendation | llm


## same thing with OPENAI

In [41]:
from openai import OpenAI

openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)

print("OpenAI client created")

OpenAI client created


In [106]:
response_openai = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": """
You are the Orchestrator of a shopping research system.

Your job is to understand a user's requirement
and break it into research tasks.

You are NOT doing the research.

Available agent types:

- browser
- llm
- logic

Use browser for website research.
Use llm for reasoning and summarization.
Use logic for deterministic calculations.
"""
        },
        {
            "role": "user",
            "content": user_requirement
        }
    ],
    text_format=TaskPlan,
)

In [44]:
task_plan_openai = response_openai.output_parsed

In [45]:
print(task_plan_openai)

user_requirement='Find the best laptop under ₹80,000 for programming, gaming, and college use.' tasks=[Task(id=1, name='Research Laptop Options', description='Find laptops that fit the budget and requirements for programming, gaming, and college use.', agent_type='browser'), Task(id=2, name='Compare Specifications', description='Analyze the specifications of shortlisted laptops including CPU, GPU, RAM, and storage capacity.', agent_type='llm'), Task(id=3, name='Evaluate User Reviews', description='Summarize user reviews and ratings for the shortlisted laptops to gauge performance in real-world scenarios.', agent_type='browser'), Task(id=4, name='Calculate Overall Value', description='Determine the overall value of each laptop by considering price, performance, and user satisfaction.', agent_type='logic')]


In [107]:
print("===== GEMINI =====")

for task in task_plan.tasks:
    print(
        task.id,
        task.name,
        "→",
        task.agent_type
    )

print("\n===== OPENAI =====")

for task in task_plan_openai.tasks:
    print(
        task.id,
        task.name,
        "→",
        task.agent_type
    )

===== GEMINI =====
1 Market Search → browser
2 Feature Analysis → llm
3 Score Calculation → logic
4 Final Recommendation → llm

===== OPENAI =====
1 Research Laptop Options → browser
2 Compare Specifications → llm
3 Evaluate User Reviews → browser
4 Calculate Overall Value → logic


In [173]:
user_requirement = """
Find the best laptop under ₹80,000 for programming,
gaming and college use.
Compare products from Amazon and Flipkart.
"""

## Return the response in `TaskPlan` format.

In [186]:
def create_plan_gemini(user_requirement: str) -> TaskPlan:
    prompt = f"""
You are the Orchestrator of a shopping research system.

Your job is to understand the user's requirement
and break it into research tasks.

You are NOT doing the research.

For each task determine:

- task id
- task name
- what the task should accomplish
- which type of agent should perform it
- URL to visit if it is a browser task

Available agent types:

browser
llm
logic

Use browser when information must be collected from websites.
Use llm for reasoning, summarization or analysis.
Use logic for deterministic calculations.

For browser tasks:
- provide the exact website URL to visit
- create separate tasks when research should be performed on different websites
- make the task description specific about what information needs to be collected

User requirement:

{user_requirement}
"""
    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": TaskPlan.model_json_schema(),
        },
    )
    return TaskPlan.model_validate_json(response.text)

In [188]:
plan = create_plan_gemini(
    "Find the best laptop under ₹80,000 for programming and gaming."
)



#### input

In [189]:
print(plan.user_requirement)

Find the best laptop under ₹80,000 for programming and gaming.


#### output of Orchestrator

In [227]:
for task in plan.tasks:
    print("ID:", task.id)
    print("NAME:", task.name)
    print("TYPE:", task.agent_type)
    print("URL:", task.url)
    print("DESCRIPTION:", task.description)
    print("-" * 50)
  

ID: 1
NAME: Search Amazon for Laptops
TYPE: browser
URL: https://www.amazon.in/s?k=laptop+under+80000+gaming
DESCRIPTION: Find gaming and programming laptops priced under ₹80,000, noting their specifications such as CPU, RAM, GPU, and price.
--------------------------------------------------
ID: 2
NAME: Search Flipkart for Laptops
TYPE: browser
URL: https://www.flipkart.com/search?q=laptop+under+80000+gaming
DESCRIPTION: Find gaming and programming laptops priced under ₹80,000, noting their specifications such as CPU, RAM, GPU, and price.
--------------------------------------------------
ID: 3
NAME: Analyze and Compare Laptops
TYPE: llm
URL: None
DESCRIPTION: Review the specifications of the laptops collected from Amazon and Flipkart, evaluate their performance for programming and gaming, and select the top options.
--------------------------------------------------
ID: 4
NAME: Calculate Value and Discounts
TYPE: logic
URL: None
DESCRIPTION: Calculate final prices, discount percentage

In [82]:
import subprocess

## fetch only those result where  `agent_type=browser`

In [228]:
import json

browser_task=[]

for task in plan.tasks:
    if task.agent_type == "browser":
        browser_task.append(task.model_dump())
        
browser_task

[{'id': 1,
  'name': 'Search Amazon for Laptops',
  'description': 'Find gaming and programming laptops priced under ₹80,000, noting their specifications such as CPU, RAM, GPU, and price.',
  'agent_type': 'browser',
  'url': 'https://www.amazon.in/s?k=laptop+under+80000+gaming'},
 {'id': 2,
  'name': 'Search Flipkart for Laptops',
  'description': 'Find gaming and programming laptops priced under ₹80,000, noting their specifications such as CPU, RAM, GPU, and price.',
  'agent_type': 'browser',
  'url': 'https://www.flipkart.com/search?q=laptop+under+80000+gaming'}]

that desciption  of finding laptops is sent in to the browser use

### creating the first_browser agent `(Amazon)`

In [ ]:
result_amazon = subprocess.run(
    [
        "python",
        "../agents/product_discovery.py",
        browser_task[0]['url'],
        browser_task[0]["description"]
    ],
    capture_output=True,
    text=True,
    encoding="utf-8"
)

now convert it to dict

In [236]:

amazon_products=json.loads(result_amazon.stdout)["products"]
print(amazon_products)

[{'name': 'ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop', 'price': 70990.0, 'rating': 4.3, 'specifications': 'AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM, 512GB SSD, FHD 15.6", Windows 11 Home', 'url': 'https://www.amazon.in/ASUS-FA506NCQ-HN006W-Gaming-Laptop/dp/B0F4J1C1V4/', 'source': 'Amazon India'}, {'name': 'ASUS TUF A15 (2025), Smartchoice, AMD Ryzen 7 7445HS, Gaming Laptop, RTX 3050-4GB, 75W TGP, 16GB RAM (Upgradeable Upto 64GB) 1TB SSD, FHD, 15.6", 144Hz, M365 Basic(1Y), Office 2024, Black, 2.3 Kg, FA506NCG-HN251WS', 'price': 77990.0, 'rating': 4.3, 'specifications': 'AMD Ryzen 7 7445HS, RTX 3050-4GB, 16GB RAM, 1TB SSD, FHD 15.6", 144Hz, Windows 11', 'url': 'https://www.amazon.in/ASUS-FA506NCG-HN251WS-Gaming-Laptop/dp/B0F4J1C1V5/', 'source': 'Amazon India'}]


### creating the second_browser agent `(Flipkart)`

In [ ]:
result_flipkart = subprocess.run(
    [
        "python",
        "../agents/product_discovery.py",
        browser_task[1]['url'],
        browser_task[1]["description"]
    ],
    capture_output=True,
    text=True,
    encoding="utf-8"
)

In [237]:
flipkart_products=json.loads(result_flipkart.stdout)["products"]
print(flipkart_products)



[{'name': 'Acer Aspire 7 (i7 14th Gen) Intel Core 7 240H - (16 GB/512 GB SSD/Windows 11 Home/6 GB Graphics/NVIDIA...', 'price': 75990.0, 'rating': 3.9, 'specifications': 'Intel Core 7 Processor, 16 GB DDR4 RAM, 64 bit Windows 11 Home Operating System, 512 GB SSD, 39.62 cm (15.6 Inch) Display, 6 GB Graphics NVIDIA', 'url': 'https://www.flipkart.com/search?q=laptop+under+80000+gaming', 'source': 'Flipkart'}, {'name': 'DELL G15 Intel Core i5 13th Gen 13450HX - (16 GB/512 GB SSD/Windows 11 Home/6 GB Graphics/NVIDIA GeFor...', 'price': 79990.0, 'rating': 4.2, 'specifications': 'Intel Core i5 Processor (13th Gen), 16 GB DDR5 RAM, Windows 11 Operating System, 512 GB SSD, 39.62 cm (15.6 Inch) Display, 6 GB Graphics NVIDIA', 'url': 'https://www.flipkart.com/search?q=laptop+under+80000+gaming', 'source': 'Flipkart'}, {'name': 'Acer Aspire 7 (i5 14th Gen) Intel Core 5 210H - (16 GB/512 GB SSD/Windows 11 Home/6 GB Graphics/NVIDIA...', 'price': 77990.0, 'rating': 4.3, 'specifications': 'Intel Core 

In [238]:
total_products=amazon_products+flipkart_products
print(total_products)

[{'name': 'ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop', 'price': 70990.0, 'rating': 4.3, 'specifications': 'AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM, 512GB SSD, FHD 15.6", Windows 11 Home', 'url': 'https://www.amazon.in/ASUS-FA506NCQ-HN006W-Gaming-Laptop/dp/B0F4J1C1V4/', 'source': 'Amazon India'}, {'name': 'ASUS TUF A15 (2025), Smartchoice, AMD Ryzen 7 7445HS, Gaming Laptop, RTX 3050-4GB, 75W TGP, 16GB RAM (Upgradeable Upto 64GB) 1TB SSD, FHD, 15.6", 144Hz, M365 Basic(1Y), Office 2024, Black, 2.3 Kg, FA506NCG-HN251WS', 'price': 77990.0, 'rating': 4.3, 'specifications': 'AMD Ryzen 7 7445HS, RTX 3050-4GB, 16GB RAM, 1TB SSD, FHD 15.6", 144Hz, Windows 11', 'url': 'https://www.amazon.in/ASUS-FA506NCG-HN251WS-Gaming-Laptop/dp/B0F4J1C1V5/', 'source': 'Amazon India'}, {'name': 'Acer Aspire 7 (i7 14th Gen) Intel Core 7 240H - (16 GB/512 GB SSD/Windows 11 Home/6 GB Graph

### convert that dict to dataframe

In [239]:
import pandas as pd

df = pd.DataFrame(total_products)

df

,name,price,rating,specifications,url,source
0,"ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 1...",70990.0,4.3,"AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM, 512GB...",https://www.amazon.in/ASUS-FA506NCQ-HN006W-Gam...,Amazon India
1,"ASUS TUF A15 (2025), Smartchoice, AMD Ryzen 7 ...",77990.0,4.3,"AMD Ryzen 7 7445HS, RTX 3050-4GB, 16GB RAM, 1T...",https://www.amazon.in/ASUS-FA506NCG-HN251WS-Ga...,Amazon India
2,Acer Aspire 7 (i7 14th Gen) Intel Core 7 240H ...,75990.0,3.9,"Intel Core 7 Processor, 16 GB DDR4 RAM, 64 bit...",https://www.flipkart.com/search?q=laptop+under...,Flipkart
3,DELL G15 Intel Core i5 13th Gen 13450HX - (16 ...,79990.0,4.2,"Intel Core i5 Processor (13th Gen), 16 GB DDR5...",https://www.flipkart.com/search?q=laptop+under...,Flipkart
4,Acer Aspire 7 (i5 14th Gen) Intel Core 5 210H ...,77990.0,4.3,"Intel Core 5 Processor, 16 GB DDR4 RAM, 64 bit...",https://www.flipkart.com/search?q=laptop+under...,Flipkart
5,HP Victus Intel Core i5 13th Gen 13420H - (16 ...,77990.0,4.4,"Intel Core i5 Processor (13th Gen), 16 GB DDR4...",https://www.flipkart.com/search?q=laptop+under...,Flipkart


In [166]:
import json

with open(
    "../outputs/amazon_products.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        products,
        f,
        indent=2,
        ensure_ascii=False
    )

## Enhancing the Product schema

In [ ]:
# from typing import List,Dict

# class Product_gen(BaseModel):
#     name:str
#     price:float|None=None
#     rating:float|None=None
#     brand:str|None=None
#     url:str|None=None
#     source:str|None=None
#     specifications:Dict[str,str]={}

#### better architecture with automatic browser agents

In [241]:
import json

browser_task=[]

for task in plan.tasks:
    if task.agent_type == "browser":
        browser_task.append(task.model_dump())
        
browser_task

[{'id': 1,
  'name': 'Search Amazon for Laptops',
  'description': 'Find gaming and programming laptops priced under ₹80,000, noting their specifications such as CPU, RAM, GPU, and price.',
  'agent_type': 'browser',
  'url': 'https://www.amazon.in/s?k=laptop+under+80000+gaming'},
 {'id': 2,
  'name': 'Search Flipkart for Laptops',
  'description': 'Find gaming and programming laptops priced under ₹80,000, noting their specifications such as CPU, RAM, GPU, and price.',
  'agent_type': 'browser',
  'url': 'https://www.flipkart.com/search?q=laptop+under+80000+gaming'}]

In [243]:
all_products = []

for task in browser_task:

    result = subprocess.run(
        [
            "python",
            "../agents/product_discovery.py",
            task["url"],
            task["description"]
        ],
        capture_output=True,
        text=True,
        encoding="utf-8"
    )

    # parse result
    products=json.loads(result.stdout)["products"]
    # add products
    all_products.extend(products)
print(all_products)    

[{'name': 'ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop', 'price': 70990.0, 'rating': 4.3, 'specifications': 'AMD Ryzen 7, RTX 3050 4GB, 16GB RAM, 512GB SSD, 15.6 inch FHD, Windows 11', 'url': 'https://www.amazon.in/dp/30683a0b-cde8-47f5-9ee6-0840a7bdb2f8', 'source': 'Amazon India'}, {'name': 'ASUS TUF A15 (2025), Smartchoice, AMD Ryzen 7 7445HS, Gaming Laptop, RTX 3050-4GB, 75W TGP, 16GB RAM (Upgradeable Upto 64GB) 1TB SSD, FHD, 15.6", 144Hz, M365 Basic(1Y), Office 2024, Black, 2.3 Kg, FA506NCG-HN251WS', 'price': 77990.0, 'rating': 4.3, 'specifications': 'AMD Ryzen 7 7445HS, RTX 3050 4GB, 16GB RAM, 1TB SSD, 15.6 inch FHD 144Hz, Windows 11', 'url': 'https://www.amazon.in/dp/ae2f60eb-1cc5-4ad0-b373-3d1ea9f89c4f', 'source': 'Amazon India'}, {'name': 'Acer Aspire 7 (i7 14th Gen) Intel Core 7 240H', 'price': 75990.0, 'rating': 3.9, 'specifications': 'Intel Core 7 Pr

In [244]:
import pandas as pd

df = pd.DataFrame(all_products)

df

,name,price,rating,specifications,url,source
0,"ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 1...",70990.0,4.3,"AMD Ryzen 7, RTX 3050 4GB, 16GB RAM, 512GB SSD...",https://www.amazon.in/dp/30683a0b-cde8-47f5-9e...,Amazon India
1,"ASUS TUF A15 (2025), Smartchoice, AMD Ryzen 7 ...",77990.0,4.3,"AMD Ryzen 7 7445HS, RTX 3050 4GB, 16GB RAM, 1T...",https://www.amazon.in/dp/ae2f60eb-1cc5-4ad0-b3...,Amazon India
2,Acer Aspire 7 (i7 14th Gen) Intel Core 7 240H,75990.0,3.9,"Intel Core 7 Processor, 16 GB DDR4 RAM, 512 GB...",https://www.flipkart.com/search?q=laptop+under...,Flipkart
3,Acer Aspire 7 (i5 14th Gen) Intel Core 5 210H,77990.0,4.3,"Intel Core 5 Processor, 16 GB DDR4 RAM, 512 GB...",https://www.flipkart.com/search?q=laptop+under...,Flipkart
4,HP Victus Intel Core i5 13th Gen 13420H,77990.0,4.4,"Intel Core i5 Processor (13th Gen), 16 GB DDR4...",https://www.flipkart.com/search?q=laptop+under...,Flipkart
5,MSI Thin 15 Intel Core i7 13th Gen 13620H,79990.0,4.5,"Intel Core i7 Processor (13th Gen), 16 GB DDR4...",https://www.flipkart.com/search?q=laptop+under...,Flipkart


In [245]:
print("Total products:", len(all_products))

Total products: 6


In [246]:
names = [product["name"] for product in all_products]

print("Total:", len(names))
print("Unique names:", len(set(names)))

Total: 6
Unique names: 6


# let's begin Engineered Normalization.

In [247]:
import re

def normalize_name(name):
    name = name.lower()

    # remove punctuation
    name = re.sub(r"[^a-z0-9\s]", " ", name)

    # remove extra spaces
    name = re.sub(r"\s+", " ", name).strip()

    return name

In [248]:
test = all_products[0]["name"]

print("Original:")
print(test)

print("\nNormalized:")
print(normalize_name(test))

Original:
ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop

Normalized:
asus tuf a15 amd ryzen 7 170 rtx 3050 4gb 16gb ram upgradeable 512gb ssd fhd 15 6 39 6 cm windows 11 home graphite black 2 3 kg fa506ncq hn006w gaming laptop


### inserted normalized name in each products

In [249]:
for product in all_products:
    product["normalized_name"] = normalize_name(product["name"])

In [250]:
all_products[0]

{'name': 'ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop',
 'price': 70990.0,
 'rating': 4.3,
 'specifications': 'AMD Ryzen 7, RTX 3050 4GB, 16GB RAM, 512GB SSD, 15.6 inch FHD, Windows 11',
 'url': 'https://www.amazon.in/dp/30683a0b-cde8-47f5-9ee6-0840a7bdb2f8',
 'source': 'Amazon India',
 'normalized_name': 'asus tuf a15 amd ryzen 7 170 rtx 3050 4gb 16gb ram upgradeable 512gb ssd fhd 15 6 39 6 cm windows 11 home graphite black 2 3 kg fa506ncq hn006w gaming laptop'}

### Data cleansing, finding same normalized names.
### A certain level of cleansing we are performiing, and after that we will send it to gemini to find duplicate prods.

In [251]:
from collections import Counter

name_counts = Counter(
    product["normalized_name"]
    for product in all_products
)

In [252]:
duplicates = {
    name: count
    for name, count in name_counts.items()
    if count > 1
}

duplicates

{}

In [257]:
def get_candidate_key(name):
    normalized = normalize_name(name)
    words = normalized.split()

    return " ".join(words[:10])

In [258]:
for product in all_products:
    print(
        get_candidate_key(product["name"]),
        "|",
        product["source"]
    )

asus tuf a15 amd ryzen 7 170 rtx 3050 4gb | Amazon India
asus tuf a15 2025 smartchoice amd ryzen 7 7445hs gaming | Amazon India
acer aspire 7 i7 14th gen intel core 7 240h | Flipkart
acer aspire 7 i5 14th gen intel core 5 210h | Flipkart
hp victus intel core i5 13th gen 13420h | Flipkart
msi thin 15 intel core i7 13th gen 13620h | Flipkart


In [259]:
from collections import defaultdict
groups = defaultdict(list)

for product in all_products:
    key = get_candidate_key(product["name"])
    groups[key].append(product)

In [260]:
for key, products in groups.items():

    print("\nGROUP:", key)

    for product in products:
        print(
            " ",
            product["source"],
            "|",
            product["name"],
            "| ₹",
            product["price"]
        )


GROUP: asus tuf a15 amd ryzen 7 170 rtx 3050 4gb
  Amazon India | ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop | ₹ 70990.0

GROUP: asus tuf a15 2025 smartchoice amd ryzen 7 7445hs gaming
  Amazon India | ASUS TUF A15 (2025), Smartchoice, AMD Ryzen 7 7445HS, Gaming Laptop, RTX 3050-4GB, 75W TGP, 16GB RAM (Upgradeable Upto 64GB) 1TB SSD, FHD, 15.6", 144Hz, M365 Basic(1Y), Office 2024, Black, 2.3 Kg, FA506NCG-HN251WS | ₹ 77990.0

GROUP: acer aspire 7 i7 14th gen intel core 7 240h
  Flipkart | Acer Aspire 7 (i7 14th Gen) Intel Core 7 240H | ₹ 75990.0

GROUP: acer aspire 7 i5 14th gen intel core 5 210h
  Flipkart | Acer Aspire 7 (i5 14th Gen) Intel Core 5 210H | ₹ 77990.0

GROUP: hp victus intel core i5 13th gen 13420h
  Flipkart | HP Victus Intel Core i5 13th Gen 13420H | ₹ 77990.0

GROUP: msi thin 15 intel core i7 13th gen 13620h
  Flipkart | MSI Thin 15 Intel Cor

## LLM as a judge for same product

In [261]:
from pydantic import BaseModel


class MatchResult(BaseModel):
    same_product: bool
    confidence: float
    reason: str

In [262]:
def match_products(product1, product2):

    prompt = f"""
You are a product matching system.

Determine whether the following two listings
refer to the SAME physical product.

Product 1:
Name: {product1["name"]}
Specifications: {product1.get("specifications")}
Source: {product1["source"]}

Product 2:
Name: {product2["name"]}
Specifications: {product2.get("specifications")}
Source: {product2["source"]}

Important:
- Ignore differences in seller/source.
- Ignore price differences.
- Ignore minor formatting differences.
- Look for model numbers, product identifiers,
  specifications and other identifying information.
- Do not assume two products are the same just
  because their names are similar.

Return:
- same_product: true or false
- confidence: number between 0 and 1
- reason: short explanation
"""
    
    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": MatchResult.model_json_schema(),
        },
    )

    return MatchResult.model_validate_json(response.text)

In [263]:
for i, product in enumerate(all_products):
    print(i, product["name"], "|", product["source"])

0 ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop | Amazon India
1 ASUS TUF A15 (2025), Smartchoice, AMD Ryzen 7 7445HS, Gaming Laptop, RTX 3050-4GB, 75W TGP, 16GB RAM (Upgradeable Upto 64GB) 1TB SSD, FHD, 15.6", 144Hz, M365 Basic(1Y), Office 2024, Black, 2.3 Kg, FA506NCG-HN251WS | Amazon India
2 Acer Aspire 7 (i7 14th Gen) Intel Core 7 240H | Flipkart
3 Acer Aspire 7 (i5 14th Gen) Intel Core 5 210H | Flipkart
4 HP Victus Intel Core i5 13th Gen 13420H | Flipkart
5 MSI Thin 15 Intel Core i7 13th Gen 13620H | Flipkart


In [273]:
product1 = all_products[0]
product2 = all_products[1]
product1

{'name': 'ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop',
 'price': 70990.0,
 'rating': 4.3,
 'specifications': 'AMD Ryzen 7, RTX 3050 4GB, 16GB RAM, 512GB SSD, 15.6 inch FHD, Windows 11',
 'url': 'https://www.amazon.in/dp/30683a0b-cde8-47f5-9ee6-0840a7bdb2f8',
 'source': 'Amazon India',
 'normalized_name': 'asus tuf a15 amd ryzen 7 170 rtx 3050 4gb 16gb ram upgradeable 512gb ssd fhd 15 6 39 6 cm windows 11 home graphite black 2 3 kg fa506ncq hn006w gaming laptop'}

In [272]:
match_result = match_products(product1, product2)

print(match_result)

same_product=True confidence=0.95 reason='Both listings share the exact same specifications (AMD Ryzen 7, RTX 3050 4GB, 16GB RAM, 512GB SSD, 15.6 inch FHD, Windows 11) and belong to the same ASUS TUF A15 product line.'
